# WEDINOS Scraper

Tutorials:
[Real Python](https://realpython.com/beautiful-soup-web-scraper-python/#dynamic-websites) 

### Single page

In [4]:
import requests
from bs4 import BeautifulSoup
import json
import re, html
import os
cwd = os.getcwd()

In [3]:
URL = "https://www.wedinos.org/sample-results"
page = requests.get(URL)

#print(page.text)
soup = BeautifulSoup(page.content, "html.parser")
alerts = soup.find_all("div", class_="alert alert-danger")
len(alerts)

20

In [3]:
data = []
for alert in alerts:
    if "The sample was not analysed" not in str(alert):
        tag_re = re.compile(r'(<!--.*?-->|<[^>]*>)')
        code = str(alert).split('<span style="font-size: 1.4em; font-weight: 700">',1)[1].split('</span>',1)[0]
        date_received = str(alert).split('Date Received: <span style="color: black">',1)[1].split('</span>',1)[0]
        postcode = str(alert).split('Postcode: <span style="color: black">',1)[1].split(' - </span>',1)[0]
        intent = str(alert).split('Purchase Intent: <span style="color: black">',1)[1].split('</span>',1)[0]
        label = str(alert).split('Package Label: <span style="color: black">',1)[1].split('</span>',1)[0]
        colour = str(alert).split('Sample Colour: <span style="color: black">',1)[1].split('</span>',1)[0]
        form = str(alert).split('Sample Form: <span style="color: black">',1)[1].split('</span>',1)[0]
        consumption_method = str(alert).split('Consumption Method: <span style="color: black">',1)[1].split('</span>',1)[0]
        effects = str(alert).split('Self-Reported Effects: <span style="color: black">',1)[1].split('</span>',1)[0]
        major = tag_re.sub('', str(alert).split('Sample Upon Analysis (Major): <span style="color: black">',1)[1].split('</span>',1)[0])
        minor = str(alert).split('Sample Upon Analysis (Minor): <span style="color: black">',1)[1].split('</span>',1)[0]
        print(f"{postcode}: Sold as {intent} ({label}), was actually {major}.")
        myAlertData = {
            "date_received": date_received,
            "postcode": postcode,
            "intent": intent,
            "label": label,
            "colour": colour,
            "form": form,
            "consumption_method": consumption_method,
            "effects": effects,
            "major": major,
            "minor": minor
        }
        data.append({code: myAlertData})

CF10: Sold as Pregabalin (Signature double), was actually Pregabalin.
LE12: Sold as Alprazolam (Rlam), was actually Alprazolam.
TR25: Sold as heroin (Not Stated), was actually Heroin, Noscapine, 6-MAM, 6-Acetylcodeine.
CH1: Sold as Xanax (Tehran Darou Alprazolam), was actually Alprazolam.
W5: Sold as Ketamine (Not Stated), was actually Ketamine.
N1: Sold as MDMA (Not Stated), was actually MDMA.
CF10: Sold as Unknown (Not Stated), was actually Cocaine.
TF9: Sold as MDMA (Not Stated), was actually MDMA.
N1: Sold as MDMA (Not Stated), was actually MDMA.
KT3: Sold as Valium (Valium Diazepam), was actually Diazepam.
DL78: Sold as Diazepam (Valium), was actually Diazepam.
AB11: Sold as Crack cocaine (Not Stated), was actually Cocaine, Phenacetin.
LE12: Sold as Diazepam (Elipam Elikem), was actually Diazepam.
SW19: Sold as CK calvin klein coke +ketamine (Not Stated), was actually MDMA, Ketamine, Cocaine, Levamisole.
TF9: Sold as Cocaine (Not Stated), was actually Cocaine.
G42: Sold as Heroin 

In [4]:
data

[{'000233645': {'date_received': '27 Mar 2025',
   'postcode': 'CF10',
   'intent': 'Pregabalin',
   'label': 'Signature double',
   'colour': 'White',
   'form': 'Capsule',
   'consumption_method': 'Oral',
   'effects': 'Relaxed, Memory Loss, Loss of consciousness',
   'major': 'Pregabalin',
   'minor': ''}},
 {'W063091': {'date_received': '27 Mar 2025',
   'postcode': 'LE12',
   'intent': 'Alprazolam',
   'label': 'Rlam',
   'colour': 'White',
   'form': 'Tablet',
   'consumption_method': 'Not Stated',
   'effects': 'Not Stated',
   'major': 'Alprazolam',
   'minor': ''}},
 {'W063231': {'date_received': '27 Mar 2025',
   'postcode': 'TR25',
   'intent': 'heroin',
   'label': 'Not Stated',
   'colour': 'Brown',
   'form': 'Powder',
   'consumption_method': 'Not Stated',
   'effects': 'Not Stated',
   'major': 'Heroin, Noscapine, 6-MAM, 6-Acetylcodeine',
   'minor': '<a data-target="#myModal" data-toggle="modal" onclick="loadiframe(\'material.php?id=00530\')" style="color: black; text-

In [ ]:
save = input("Save to wedinos_data.json? (y/n): ")
if save.lower() == 'y':
    # Save the data to a JSON file
    with open('data/wedinos_data.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

## Dynamic scraping

In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

In [8]:
driver = webdriver.Chrome()
driver.get("https://wedinos.org/sample-results")

current_page = 0
max_pages = 28 #367 was number of pages for 1 Jan to 4 Dec 2024
all_pages = []

while current_page < max_pages:
    try:
        # After loading all items, scrape the data
        all_pages.append(driver.page_source)
            
        load_more_button = driver.find_element(By.XPATH, "//a[text()='Next']")
        load_more_button.click()
        time.sleep(3)  # Give time for content to load
        current_page += 1
    except:
        break

driver.quit()

In [9]:
all_alerts = []
for page in all_pages:
    soup = BeautifulSoup(page, "html.parser")
    alerts = soup.find_all("div", class_="alert alert-danger")
    #all_alerts.append(alerts)
    for alert in alerts:
        try:
            tag_re = re.compile(r'(<!--.*?-->|<[^>]*>)')
            code = str(alert).split('<span style="font-size: 1.4em; font-weight: 700">',1)[1].split('</span>',1)[0]
            date_received = str(alert).split('Date Received: <span style="color: black">',1)[1].split('</span>',1)[0]
            postcode = str(alert).split('Postcode: <span style="color: black">',1)[1].split(' - </span>',1)[0]
            intent = str(alert).split('Purchase Intent: <span style="color: black">',1)[1].split('</span>',1)[0]
            label = str(alert).split('Package Label: <span style="color: black">',1)[1].split('</span>',1)[0]
            colour = str(alert).split('Sample Colour: <span style="color: black">',1)[1].split('</span>',1)[0]
            form = str(alert).split('Sample Form: <span style="color: black">',1)[1].split('</span>',1)[0]
            consumption_method = str(alert).split('Consumption Method: <span style="color: black">',1)[1].split('</span>',1)[0]
            effects = str(alert).split('Self-Reported Effects: <span style="color: black">',1)[1].split('</span>',1)[0]
            major = tag_re.sub('', str(alert).split('Sample Upon Analysis (Major): <span style="color: black">',1)[1].split('</span>',1)[0])
            minor = tag_re.sub('', str(alert).split('Sample Upon Analysis (Minor): <span style="color: black">',1)[1].split('</span>',1)[0])
            print(f"{postcode}: Sold as {intent} ({label}), was actually {major}.")
            myAlertData = {
                "date_received": date_received,
                "postcode": postcode,
                "intent": intent,
                "label": label,
                "colour": colour,
                "form": form,
                "consumption_method": consumption_method,
                "effects": effects,
                "major": major,
                "minor": minor
            }
            all_alerts.append({code: myAlertData})
        except:
            pass

CF10: Sold as Pregabalin (Signature double), was actually Pregabalin.
LE12: Sold as Alprazolam (Rlam), was actually Alprazolam.
TR25: Sold as heroin (Not Stated), was actually Heroin, Noscapine, 6-MAM, 6-Acetylcodeine.
CH1: Sold as Xanax (Tehran Darou Alprazolam), was actually Alprazolam.
W5: Sold as Ketamine (Not Stated), was actually Ketamine.
N1: Sold as MDMA (Not Stated), was actually MDMA.
CF10: Sold as Unknown (Not Stated), was actually Cocaine.
TF9: Sold as MDMA (Not Stated), was actually MDMA.
N1: Sold as MDMA (Not Stated), was actually MDMA.
KT3: Sold as Valium (Valium Diazepam), was actually Diazepam.
DL78: Sold as Diazepam (Valium), was actually Diazepam.
AB11: Sold as Crack cocaine (Not Stated), was actually Cocaine, Phenacetin.
LE12: Sold as Diazepam (Elipam Elikem), was actually Diazepam.
SW19: Sold as CK calvin klein coke +ketamine (Not Stated), was actually MDMA, Ketamine, Cocaine, Levamisole.
TF9: Sold as Cocaine (Not Stated), was actually Cocaine.
G42: Sold as Heroin 

In [ ]:
save_alerts = input('Do you want to save the alerts to a file? (y/n): ')
if save_alerts == 'y':
    alerts_name = input('Enter the name of the file to save the alerts to: ')
    with open('data/'+alerts_name+'.json', 'w', encoding='utf-8') as f:
        json.dump(all_alerts, f, ensure_ascii=False, indent=4)
#with open('wedinos_alerts_2024.json', 'w', encoding='utf-8') as f:
    #json.dump(all_alerts, f, ensure_ascii=False, indent=4)

In [6]:
open_alerts = input('Do you want to open a saved alerts file? (y/n): ')
if open_alerts == 'y':
    alerts_name = input('Enter the name of the json file to open: ')
    with open('data/'+alerts_name+'.json', 'r', encoding='utf-8') as f:
        all_alerts = json.load(f)
        print(len(all_alerts))
#with open('wedinos_alerts_2024.json', 'r', encoding='utf-8') as f:
    #all_alerts = json.load(f)
    #print(len(all_alerts))

6025


In [7]:
for alert in all_alerts:
    for i in alert:
        if ('nitazene' in alert[i]['major']) or ('nitazepipne' in alert[i]['major']) or ('nitazepyne' in alert[i]['major']):
            print(i,alert[i]['date_received'], alert[i]['postcode'],'– Sold as', alert[i]['intent'], ', tested as',alert[i]['major'], 'with' if(len(alert[i]['minor'])>=1) else '', alert[i]['minor'])
        elif ('nitazene' in alert[i]['minor']) or ('nitazepipne' in alert[i]['minor']) or ('nitazepyne' in alert[i]['minor']):
            print(i,alert[i]['date_received'], alert[i]['postcode'],'– Sold as', alert[i]['intent'], ', tested as',alert[i]['major'], 'with' if(len(alert[i]['minor'])>=1) else '', alert[i]['minor'])
        elif ('nitazene' in alert[i]['intent']) or ('nitazepipne' in alert[i]['intent']) or ('nitazepyne' in alert[i]['intent']):
            print(i,alert[i]['date_received'], alert[i]['postcode'],'– Sold as', alert[i]['intent'], ', tested as',alert[i]['major'], 'with' if(len(alert[i]['minor'])>=1) else '', alert[i]['minor'])
        elif ('methazene' in alert[i]['major']) or ('methazene' in alert[i]['minor']) or ('methazene' in alert[i]['intent']):
            print(i,'!!!!!!!')

W059957 26 Nov 2024 BN3 – Sold as Diazepam , tested as Bromazolam, Metonitazene  
000230437 21 Nov 2024 NP20 – Sold as Heroin , tested as xylazine with Paracetamol, Metonitazene, Caffeine, Bromazolam
W059740 19 Nov 2024 NE66 – Sold as Isotonitazene , tested as Isotonitazepyne  
000233224 14 Nov 2024 CF10 – Sold as HEROIN , tested as Noscapine, Caffeine, Paracetamol, Heroin, xylazine, 6-MAM, 6-Acetylcodeine with Papaverine, Metonitazene
W059583 14 Nov 2024 BS4 – Sold as diazepam , tested as Bromazolam, Metonitazene  
000233570 20 Nov 2024 NP10 – Sold as Heroin , tested as xylazine with Paracetamol, Metonitazene, Noscapine
W059402 12 Nov 2024 SA1 – Sold as Heroin , tested as xylazine with Paracetamol, Caffeine, Metonitazene
W059171 05 Nov 2024 BS4 – Sold as Diazepam , tested as Metonitazene, Bromazolam  
W059041 05 Nov 2024 TQ46 – Sold as Diazepam , tested as Bromazolam, Metonitazene  
W058796 22 Oct 2024 W7 – Sold as Oxy , tested as Metonitazene  
W054692 15 Oct 2024 EH26 – Sold as Benz

## Postcodes

In [8]:
import pandas as pd
import folium
from folium import plugins
import pgeocode
nomi = pgeocode.Nominatim('gb')

In [9]:
#col = list(myAlertData.keys())
col=["date_received","postcode","intent","label","colour","form","consumption_method","effects","major","minor"]
col.append('latitude')
col.append('longitude')
print(col)

['date_received', 'postcode', 'intent', 'label', 'colour', 'form', 'consumption_method', 'effects', 'major', 'minor', 'latitude', 'longitude']


In [10]:
str(nomi.query_postal_code('TQ46')['latitude'])

'nan'

In [11]:
##### BENZO, NOT NITAZENE ####
df = pd.DataFrame(columns=col)
for alert in all_alerts:
    for i in alert:
        if ('azepine' in alert[i]['major']) or ('zolam' in alert[i]['major']) or ('azepine' in alert[i]['minor']) or ('zolam' in alert[i]['minor']) or ('azepine' in alert[i]['intent']) or ('zolam' in alert[i]['intent']) or ('xanax' in alert[i]['intent']) or ('diaze' in alert[i]['intent']) or ('benzo' in alert[i]['intent']) or ('MSJ' in alert[i]['intent']):
            if str(nomi.query_postal_code(alert[i]['postcode'])['latitude']) != 'nan':
                lat, long = float(nomi.query_postal_code(alert[i]['postcode'])['latitude']), float(nomi.query_postal_code(alert[i]['postcode'])['longitude'])
            else:
                pcode = alert[i]['postcode'][:3]
                lat, long = float(nomi.query_postal_code(pcode)['latitude']), float(nomi.query_postal_code(pcode)['longitude'])

            alert[i]['latitude'] = lat
            alert[i]['longitude'] = long
            df.loc[i] = pd.Series(alert[i])
            print(i,alert[i]['date_received'], alert[i]['postcode'],'– Sold as', alert[i]['intent'], ', tested as',alert[i]['major'], 'with' if(len(alert[i]['minor'])>=1) else '', alert[i]['minor'])
        

W059966 26 Nov 2024 LE11 – Sold as Alprazolam , tested as Alprazolam  
W059965 26 Nov 2024 LE11 – Sold as Midazolam , tested as Midazolam  
W059963 26 Nov 2024 LE11 – Sold as Alprazolam , tested as Alprazolam  
W060023 26 Nov 2024 LS18 – Sold as Alprazolam , tested as Alprazolam  
W059931 26 Nov 2024 WS15 – Sold as Rlam , tested as Alprazolam  
W059827 26 Nov 2024 LS12 – Sold as Valium , tested as Bromazolam  
W059828 26 Nov 2024 LS12 – Sold as Xanax , tested as Caffeine, Bromazolam  
000235000 26 Nov 2024 CF37 – Sold as Diazepam , tested as Bromazolam  
000038113 26 Nov 2024 CF72 – Sold as Found , tested as Bromazolam  
W058363 26 Nov 2024 PH11 – Sold as Valium , tested as Paracetamol, Etizolam  
000234540 26 Nov 2024 CF10 – Sold as Xanax , tested as Alprazolam  
000037601 26 Nov 2024 NP22 – Sold as Alprazolam , tested as Alprazolam  
000234544 26 Nov 2024 CF10 – Sold as Diazepam , tested as Bromazolam  
000232957 26 Nov 2024 CF36 – Sold as Diazepam , tested as Paracetamol, Bromazolam

In [12]:
#all_alerts = json.load(open('wedinos_alerts_2024.json')) # remove if you want to scrape the data again
df = pd.DataFrame(columns=col)
for alert in all_alerts:
    for i in alert:
        if ('nitazene' in alert[i]['major']) or ('nitazepipne' in alert[i]['major']) or ('nitazepyne' in alert[i]['major']) or ('nitazene' in alert[i]['minor']) or ('nitazepipne' in alert[i]['minor']) or ('nitazepyne' in alert[i]['minor']) or ('nitazene' in alert[i]['intent']) or ('nitazepipne' in alert[i]['intent']) or ('nitazepyne' in alert[i]['intent']):
            if str(nomi.query_postal_code(alert[i]['postcode'])['latitude']) != 'nan':
                lat, long = float(nomi.query_postal_code(alert[i]['postcode'])['latitude']), float(nomi.query_postal_code(alert[i]['postcode'])['longitude'])
            else:
                pcode = alert[i]['postcode'][:3]
                lat, long = float(nomi.query_postal_code(pcode)['latitude']), float(nomi.query_postal_code(pcode)['longitude'])

            alert[i]['latitude'] = lat
            alert[i]['longitude'] = long
            df.loc[i] = pd.Series(alert[i])
            print(i,alert[i]['date_received'], alert[i]['postcode'],'– Sold as', alert[i]['intent'], ', tested as',alert[i]['major'], 'with' if(len(alert[i]['minor'])>=1) else '', alert[i]['minor'])
        

W059957 26 Nov 2024 BN3 – Sold as Diazepam , tested as Bromazolam, Metonitazene  
000230437 21 Nov 2024 NP20 – Sold as Heroin , tested as xylazine with Paracetamol, Metonitazene, Caffeine, Bromazolam
W059740 19 Nov 2024 NE66 – Sold as Isotonitazene , tested as Isotonitazepyne  
000233224 14 Nov 2024 CF10 – Sold as HEROIN , tested as Noscapine, Caffeine, Paracetamol, Heroin, xylazine, 6-MAM, 6-Acetylcodeine with Papaverine, Metonitazene
W059583 14 Nov 2024 BS4 – Sold as diazepam , tested as Bromazolam, Metonitazene  
000233570 20 Nov 2024 NP10 – Sold as Heroin , tested as xylazine with Paracetamol, Metonitazene, Noscapine
W059402 12 Nov 2024 SA1 – Sold as Heroin , tested as xylazine with Paracetamol, Caffeine, Metonitazene
W059171 05 Nov 2024 BS4 – Sold as Diazepam , tested as Metonitazene, Bromazolam  
W059041 05 Nov 2024 TQ46 – Sold as Diazepam , tested as Bromazolam, Metonitazene  
W058796 22 Oct 2024 W7 – Sold as Oxy , tested as Metonitazene  
W054692 15 Oct 2024 EH26 – Sold as Benz

In [13]:
save_table = input('Do you want to save the table? (y/n): ')
if save_table == 'y':
    table_name = input('Enter the name of the table: ')
    df.to_csv('/data/'+table_name+'.csv', sep='\t', encoding='utf-8')
    #df.to_csv('wedinos_alerts_2024.csv', sep='\t', encoding='utf-8')
df

,date_received,postcode,intent,label,colour,form,consumption_method,effects,major,minor,latitude,longitude
W059957,26 Nov 2024,BN3,Diazepam,Not Stated,"Purple, Black",Tablet,Not Stated,Not Stated,"Bromazolam, Metonitazene",,50.836725,-0.179625
000230437,21 Nov 2024,NP20,Heroin,Not Stated,Brown,Solid,Smoked,Loss of consciousness,xylazine,"Paracetamol, Metonitazene, Caffeine, Bromazolam",51.5877,-2.9984
W059740,19 Nov 2024,NE66,Isotonitazene,Not Stated,"Colourless, Yellow",Powder,Intravenous,"Euphoria, Empathy, Relaxed",Isotonitazepyne,,55.419943,-1.790742
000233224,14 Nov 2024,CF10,HEROIN,Not Stated,Brown,Powder,Not Stated,Not Stated,"Noscapine, Caffeine, Paracetamol, Heroin, xyla...","Papaverine, Metonitazene",51.48,-3.18
W059583,14 Nov 2024,BS4,diazepam,diazepam accord 10mg,Blue,Tablet,Not Stated,Not Stated,"Bromazolam, Metonitazene",,51.4345,-2.55525
...,...,...,...,...,...,...,...,...,...,...,...,...
W046706,10 Jan 2024,S75,Diazepam,Diazepam,"Blue, Blue",Tablet,Oral,"Increased Confidence, Relaxed, Memory Loss","Metonitazene, Bromazolam",,53.560985,-1.539131
W044688,04 Jan 2024,NG92,Diazepam,Not Stated,White,Tablet,Oral,"Euphoria, Relaxed","Metonitazene, Bromazolam, Heroin, Noscapine","Paracetamol, Caffeine",52.91914,-1.24548
000040892,04 Jan 2024,CF24,Heroin,Heroin,Brown,Powder,Intravenous,Overdose,Protonitazepyne,Caffeine,51.48,-3.18
W044755,22 Dec 2023,NE29,Zopiclone,Not Stated,White,Tablet,Not Stated,Not Stated,"Metonitazene, Bromazolam",,55.0165,-1.4493


## Plot map

In [14]:
# Initialize map
m = folium.Map(
    location=[53.989955, -3.151694],  # center of the map
    zoom_start=5,  # dezoom
    tiles='cartodb positron'  # background style
)
marker_cluster = plugins.MarkerCluster(name="nitazenes").add_to(m)
benzo_cluster = plugins.MarkerCluster(name="counterfeit_benzos").add_to(m)
# Add all the individual earthquakes to the map
for idx, row in df.iterrows():
    if len(row['minor']) ==0:
        #popup = f"{row['postcode']} – Sold as {row['intent']}, tested as {row['major']}"
        popup = f"""
            <h1>{idx}</h1>
            <p>
            Postcode: <b>{row['postcode']}</b><br/>
            Date: <b>{row['date_received']}</b><br/>
            Sold as: <b>{row['intent']}</b><br/>
            Tested as: <b>{row['major']}</b><br/>
            </p>
            """

    else:
        #popup = f"{row['postcode']} – Sold as {row['intent']}, tested as {row['major']} with {row['minor']}"
        popup = f"""
            <h1>{idx}</h1>
            <p>
            Postcode: <b>{row['postcode']}</b><br/>
            Date: <b>{row['date_received']}</b><br/>
            Sold as: <b>{row['intent']}</b><br/>
            Tested as: <b>{row['major']}</b> with <b>{row['minor']}</b><br/>
            </p>
            """
    
    color = '#1e3d77' if ('diazepam' in str.lower(row['intent'])) or ('temazepam' in str.lower(row['intent'])) or ('bromazolam' in str.lower(row['intent'])) or ('etizolam' in str.lower(row['intent'])) or ('valium' in str.lower(row['intent'])) or ('alprazolam' in str.lower(row['intent'])) or ('xanax' in str.lower(row['intent'])) or ('bensedin' in str.lower(row['intent'])) or ('msj' in str.lower(row['intent'])) or ('benzodine' in str.lower(row['intent'])) else '#ffde5b'
    try:
        if ('diazepam' in str.lower(row['intent'])) or ('temazepam' in str.lower(row['intent'])) or ('bromazolam' in str.lower(row['intent'])) or ('etizolam' in str.lower(row['intent'])) or ('valium' in str.lower(row['intent'])) or ('alprazolam' in str.lower(row['intent'])) or ('xanax' in str.lower(row['intent'])) or ('bensedin' in str.lower(row['intent'])) or ('msj' in str.lower(row['intent'])) or ('benzodine' in str.lower(row['intent'])):
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=15,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.5,
                weight=1,
                popup=popup
            ).add_to(benzo_cluster)
        else:
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=15,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.5,
                weight=1,
                popup=popup
            ).add_to(marker_cluster)
    except:
        pass

m

In [15]:
save_map = input('Do you want to save the map? (y/n): ')
if save_map == 'y':
    map_name = input('Enter the name of the map: ')
    m.save(f'/{cwd}/data/{map_name}.html')

### Other calculations

In [16]:
cbenzo_count=0
cbenzo_meto_count = 0
total=0
for idx, row in df.iterrows():
    if ('diazepam' in str.lower(row['intent'])) or ('temazepam' in str.lower(row['intent'])) or ('valium' in str.lower(row['intent'])) or ('alprazolam' in str.lower(row['intent'])) or ('xanax' in str.lower(row['intent'])) or ('bensedin' in str.lower(row['intent'])) or ('msj' in str.lower(row['intent'])) or ('benzodine' in str.lower(row['intent'])):
        cbenzo_count+=1
        total+=1
        if ('metonitazene' in str.lower(row['major'])) or ('metonitazene' in str.lower(row['minor'])):
            cbenzo_meto_count+=1
        else:
            print(row['major'],row['minor'])
    else:
        total+=1
print(cbenzo_count,'out of',total)
print(100*cbenzo_count/total,'%')
print(cbenzo_meto_count,'out of',cbenzo_count,'contained metonitazene (',100*cbenzo_meto_count/cbenzo_count,'%)')

Bromazolam, Protonitazene, N-desethylprotonitazene 
Protonitazene 
77 out of 144
53.47222222222222 %
75 out of 77 contained metonitazene ( 97.40259740259741 %)


In [17]:
msjb_count=0
total=0
for idx, row in df.iterrows():
    if ('bensedin' in str.lower(row['intent'])) or ('msj' in str.lower(row['intent'])) or ('benzodine' in str.lower(row['intent'])):
        msjb_count+=1
        total+=1
    else:
        total+=1
print(msjb_count,'intended as MSJ/Bensedin/Benzodine')
print(100*msjb_count/total,'%')

6 intended as MSJ/Bensedin/Benzodine
4.166666666666667 %
